## Import library

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings
import optbinning
import pickle
import joblib

pd.set_option("display.max_columns", 500)
pd.set_option("display.max_rows", 500)
warnings.filterwarnings("ignore")

import sys
sys.path.append("../src")

from data_eda.data_eda import *
from data_eda.plot import * 
from features.features_eng import *
from models.model_training import *
from utils.logger import *

(CVXPY) Mar 09 03:03:00 PM: Encountered unexpected exception importing solver GLOP:
RuntimeError('Unrecognized new version of ortools (9.15.6755). Expected < 9.15.0. Please open a feature request on cvxpy to enable support for this version.')
(CVXPY) Mar 09 03:03:00 PM: Encountered unexpected exception importing solver PDLP:
RuntimeError('Unrecognized new version of ortools (9.15.6755). Expected < 9.15.0. Please open a feature request on cvxpy to enable support for this version.')


## Read dataset

In [2]:
data = pd.read_csv("../data/processed/10K_Lending_Club_Loans_optbinning.csv")
data_before_bin = pd.read_csv("../data/interim/10K_Lending_Club_Loans_final_features.csv")

display(data.head(5))
display(data_before_bin.head(5))

,loan_amnt,term,int_rate,grade,annual_inc,verification_status,purpose,inq_last_6mths,revol_util,total_acc,loan_amnt_per_installment,income_to_interest_ratio,is_bad
0,0.149256,-0.584250,1.078203,0.810092,-0.019761,0.181547,-0.728222,0.140039,0.414956,0.279415,0.03658,0.245598,0
1,-0.113627,-0.584250,-0.451049,-0.670379,-0.256405,0.181547,-0.064651,-0.131600,-0.076787,-0.263539,0.03658,-0.765318,0
2,0.069937,0.274193,1.078203,0.810092,-0.019761,0.181547,0.327709,0.140039,0.414956,-0.175585,0.03658,0.245598,0
3,-0.113627,-0.584250,0.274596,0.226565,-0.019761,0.181547,-0.064651,0.140039,0.175551,0.046724,0.03658,-0.457504,0
4,-0.113627,0.274193,0.274596,0.226565,-0.019761,-0.175415,-0.064651,-0.131600,0.175551,0.046724,0.03658,-0.457504,0


,loan_amnt,term,int_rate,grade,annual_inc,verification_status,purpose,inq_last_6mths,revol_util,total_acc,is_bad,loan_amnt_per_installment,income_to_interest_ratio
0,4000,60 months,0.0729,A,50000.0,not verified,medical,0.0,12.1,44.0,0,50.150451,12.500000
1,16000,60 months,0.1825,F,39216.0,not verified,debt_consolidation,2.0,64.0,5.0,0,39.169604,2.451000
2,8700,36 months,0.0788,A,65000.0,not verified,credit_card,0.0,0.6,8.0,0,31.967665,7.471264
3,18000,60 months,0.1149,B,57500.0,not verified,debt_consolidation,0.0,37.1,23.0,0,45.479812,3.194444
4,16000,36 months,0.1183,B,50004.0,VERIFIED - income,debt_consolidation,4.0,40.4,21.0,0,30.180138,3.125250


## Train test split

In [3]:
X_train, X_test, y_train, y_test = stratified_train_test_split(data.drop(columns = ['is_bad']), y = data['is_bad'], test_size=0.2, random_state=285)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(8000, 12) (2000, 12) (8000,) (2000,)


## Model training

### Logistic Regression

In [4]:
log_model, log_study, log_res = tune_logistic(X_train, y_train, X_test, y_test)

[I 2026-03-09 15:03:01,149] A new study created in memory with name: no-name-546094d4-13a3-476d-b3ac-ae1a073b1b2e
[I 2026-03-09 15:03:01,170] Trial 0 finished with value: 0.6770491887728447 and parameters: {'C': 0.0011801017788447591}. Best is trial 0 with value: 0.6770491887728447.
[I 2026-03-09 15:03:01,189] Trial 1 finished with value: 0.6818896249642001 and parameters: {'C': 0.0032765039783292796}. Best is trial 1 with value: 0.6818896249642001.
[I 2026-03-09 15:03:01,208] Trial 2 finished with value: 0.6860343556890942 and parameters: {'C': 0.006837770185746756}. Best is trial 2 with value: 0.6860343556890942.
[I 2026-03-09 15:03:01,230] Trial 3 finished with value: 0.6937940323221801 and parameters: {'C': 0.027427668174717125}. Best is trial 3 with value: 0.6937940323221801.
[I 2026-03-09 15:03:01,254] Trial 4 finished with value: 0.697584690642391 and parameters: {'C': 0.3841959813181318}. Best is trial 4 with value: 0.697584690642391.
[I 2026-03-09 15:03:01,282] Trial 5 finishe

### XGBoost

In [5]:
xgb_model, xgb_study, xgb_res = tune_xgb(X_train, y_train, X_test, y_test)

[I 2026-03-09 15:03:02,625] A new study created in memory with name: no-name-16fb6798-e512-4e6c-a791-0ea35033ecf9
[I 2026-03-09 15:03:04,412] Trial 0 finished with value: 0.6052080132573756 and parameters: {'n_estimators': 484, 'max_depth': 6, 'learning_rate': 0.1090941184894473, 'subsample': 0.718526320944598, 'colsample_bytree': 0.9265591303016426}. Best is trial 0 with value: 0.6052080132573756.
[I 2026-03-09 15:03:05,715] Trial 1 finished with value: 0.6003628316053182 and parameters: {'n_estimators': 463, 'max_depth': 6, 'learning_rate': 0.14405488894369525, 'subsample': 0.6225567893004166, 'colsample_bytree': 0.9398004723815622}. Best is trial 0 with value: 0.6052080132573756.
[I 2026-03-09 15:03:06,231] Trial 2 finished with value: 0.6432194234035942 and parameters: {'n_estimators': 451, 'max_depth': 4, 'learning_rate': 0.08719590653303903, 'subsample': 0.6781918218503674, 'colsample_bytree': 0.9191096026243759}. Best is trial 2 with value: 0.6432194234035942.
[I 2026-03-09 15:0

### LGBM

In [6]:
lgb_model, lgb_study, lgb_res = tune_lgbm(X_train, y_train, X_test, y_test)

[I 2026-03-09 15:03:30,358] A new study created in memory with name: no-name-8209edfa-e488-45c8-b506-d0968a01fdeb


[LightGBM] [Info] Number of positive: 829, number of negative: 5571
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000766 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 62
[LightGBM] [Info] Number of data points in the train set: 6400, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.129531 -> initscore=-1.905110
[LightGBM] [Info] Start training from score -1.905110
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -in

[I 2026-03-09 15:03:31,002] Trial 0 finished with value: 0.6909373028928594 and parameters: {'n_estimators': 248, 'max_depth': 3, 'learning_rate': 0.020833096323649436, 'subsample': 0.8548835433278651, 'colsample_bytree': 0.6968575720578781}. Best is trial 0 with value: 0.6909373028928594.


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[I 2026-03-09 15:03:31,730] Trial 1 finished with value: 0.6503623219703709 and parameters: {'n_estimators': 372, 'max_depth': 3, 'learning_rate': 0.1860265258038538, 'subsample': 0.7998711276611022, 'colsample_bytree': 0.7804121221183972}. Best is trial 0 with value: 0.6909373028928594.


...

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[I 2026-03-09 15:04:18,441] Trial 48 finished with value: 0.6867875211160228 and parameters: {'n_estimators': 180, 'max_depth': 4, 'learning_rate': 0.02051639164185071, 'subsample': 0.8516433505421225, 'colsample_bytree': 0.8470089508205029}. Best is trial 47 with value: 0.6913810279642263.


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[I 2026-03-09 15:04:18,798] Trial 49 finished with value: 0.6726480497420002 and parameters: {'n_estimators': 140, 'max_depth': 3, 'learning_rate': 0.19440372130161968, 'subsample': 0.91378838185044, 'colsample_bytree': 0.6298609951085239}. Best is trial 47 with value: 0.6913810279642263.


[LightGBM] [Info] Number of positive: 829, number of negative: 5571
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000226 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 62
[LightGBM] [Info] Number of data points in the train set: 6400, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.129531 -> initscore=-1.905110
[LightGBM] [Info] Start training from score -1.905110
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -in

### Catboost

In [7]:
# cat_model, cat_study, cat_res = tune_catboost(X_train, y_train, X_test, y_test)

## Best Model

In [8]:
results = pd.DataFrame([
    log_res,
    xgb_res,
    lgb_res,
    # cat_res
])

results

,model,train_auc,test_auc,train_ks,test_ks
0,logistic,0.700651,0.681938,0.297397,0.277822
1,xgboost,0.726186,0.672308,0.329586,0.279897
2,lightgbm,0.721636,0.663556,0.326627,0.263300


In [9]:
log_model

LogisticRegression(C=0.5764922915002083, max_iter=2000, solver='liblinear')

## Get prediction

In [10]:
wholebase_predict = log_model.predict_proba(data.drop(columns = ['is_bad']))[:, 1]
data_result = pd.concat([data, pd.Series(wholebase_predict, name='prediction')], axis=1)
data_before_bin_result = pd.concat([data_before_bin, pd.Series(wholebase_predict, name='prediction')], axis=1)

display(data_result.head(5))
display(data_before_bin_result.head(5))

,loan_amnt,term,int_rate,grade,annual_inc,verification_status,purpose,inq_last_6mths,revol_util,total_acc,loan_amnt_per_installment,income_to_interest_ratio,is_bad,prediction
0,0.149256,-0.584250,1.078203,0.810092,-0.019761,0.181547,-0.728222,0.140039,0.414956,0.279415,0.03658,0.245598,0,0.095670
1,-0.113627,-0.584250,-0.451049,-0.670379,-0.256405,0.181547,-0.064651,-0.131600,-0.076787,-0.263539,0.03658,-0.765318,0,0.372553
2,0.069937,0.274193,1.078203,0.810092,-0.019761,0.181547,0.327709,0.140039,0.414956,-0.175585,0.03658,0.245598,0,0.032019
3,-0.113627,-0.584250,0.274596,0.226565,-0.019761,0.181547,-0.064651,0.140039,0.175551,0.046724,0.03658,-0.457504,0,0.145895
4,-0.113627,0.274193,0.274596,0.226565,-0.019761,-0.175415,-0.064651,-0.131600,0.175551,0.046724,0.03658,-0.457504,0,0.125060


,loan_amnt,term,int_rate,grade,annual_inc,verification_status,purpose,inq_last_6mths,revol_util,total_acc,is_bad,loan_amnt_per_installment,income_to_interest_ratio,prediction
0,4000,60 months,0.0729,A,50000.0,not verified,medical,0.0,12.1,44.0,0,50.150451,12.500000,0.095670
1,16000,60 months,0.1825,F,39216.0,not verified,debt_consolidation,2.0,64.0,5.0,0,39.169604,2.451000,0.372553
2,8700,36 months,0.0788,A,65000.0,not verified,credit_card,0.0,0.6,8.0,0,31.967665,7.471264,0.032019
3,18000,60 months,0.1149,B,57500.0,not verified,debt_consolidation,0.0,37.1,23.0,0,45.479812,3.194444,0.145895
4,16000,36 months,0.1183,B,50004.0,VERIFIED - income,debt_consolidation,4.0,40.4,21.0,0,30.180138,3.125250,0.125060


## Export

In [11]:
data_result.to_csv("../data/predicted/data10K_Lending_Club_Loans_predicted_bin.csv", index = False)

In [12]:
data_before_bin_result.to_csv("../data/predicted/data10K_Lending_Club_Loans_predicted.csv", index = False)

In [13]:
feature_use = data_result.drop(columns = ['is_bad', 'prediction']).columns

joblib.dump(feature_use, "../data/artifacts/feature_use.pkl")

['../data/artifacts/feature_use.pkl']

In [14]:
joblib.dump(log_model, "../data/artifacts/logistic_model.pkl")

['../data/artifacts/logistic_model.pkl']

## Test prediction for deployment

In [15]:
test = pd.read_csv("../data/raw/10K_Lending_Club_Loans.csv", encoding='ISO-8859-1')
test = test.iloc[[0],:]

test

,loan_amnt,funded_amnt,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,pymnt_plan,url,desc,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,mths_since_last_major_derog,policy_code,is_bad
0,4000,4000,60 months,7.29%,79.76,A,A4,Time Warner Cable,10+ years,MORTGAGE,50000.0,not verified,n,https://www.lendingclub.com/browse/loanDetail....,NaN,medical,Medical,766xx,TX,10.87,0.0,12/1/92,0.0,NaN,NaN,15.0,0.0,12087,12.1,44.0,f,NaN,1,0


In [16]:
def pre_process(data):
    data_use = data.copy()
    data_use['int_rate'] = data_use['int_rate'].str.replace('%', '').astype(float) / 100
    data_use['loan_amnt_per_installment'] = data_use['loan_amnt'] / data_use['installment']
    data_use['income_to_interest_ratio'] = data_use['annual_inc'] / data_use['loan_amnt']

    return data_use

def binning(data):
    data_use = data.copy()
    binning = joblib.load("../data/artifacts/optbinning.pkl")
    feature_use = joblib.load("../data/artifacts/feature_use.pkl")

    data_use = data_use[feature_use]
    data_use = binning.transform(data_use)

    return data_use

def predict(data):
    data_use = data.copy()
    model = joblib.load("../data/artifacts/logistic_model.pkl")
    prediction = model.predict_proba(data_use)[:, 1]
    data_use = pd.concat([data_use, pd.Series(prediction, name="prediction")], axis=1)

    return data_use

test = pre_process(test)
display(test)

test = binning(test)
display(test)

test = predict(test)
display(test)

,loan_amnt,funded_amnt,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,pymnt_plan,url,desc,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,mths_since_last_major_derog,policy_code,is_bad,loan_amnt_per_installment,income_to_interest_ratio
0,4000,4000,60 months,0.0729,79.76,A,A4,Time Warner Cable,10+ years,MORTGAGE,50000.0,not verified,n,https://www.lendingclub.com/browse/loanDetail....,NaN,medical,Medical,766xx,TX,10.87,0.0,12/1/92,0.0,NaN,NaN,15.0,0.0,12087,12.1,44.0,f,NaN,1,0,50.150451,12.5


,loan_amnt,term,int_rate,grade,annual_inc,verification_status,purpose,inq_last_6mths,revol_util,total_acc,loan_amnt_per_installment,income_to_interest_ratio
0,0.149256,-0.58425,1.078203,0.810092,-0.019761,0.181547,-0.728222,0.140039,0.414956,0.279415,0.03658,0.245598


,loan_amnt,term,int_rate,grade,annual_inc,verification_status,purpose,inq_last_6mths,revol_util,total_acc,loan_amnt_per_installment,income_to_interest_ratio,prediction
0,0.149256,-0.58425,1.078203,0.810092,-0.019761,0.181547,-0.728222,0.140039,0.414956,0.279415,0.03658,0.245598,0.09567
